## Polygonization Exercise

In [ ]:
from pygel3d import hmesh, jupyter_display as jd
jd.set_export_mode(True)
import numpy as np
from numpy.linalg import norm
from math import exp,floor
array = np.array
        
cube_faces = np.array([[[0,-0.5,-0.5],[0,0.5,-0.5],[0,0.5,0.5],[0,-0.5,0.5]],
    [[0, 0.5,-0.5],[0,-0.5,-0.5],[0,-0.5,0.5],[0,0.5,0.5]],
    [[ 0.5,0, -0.5],[-0.5,0, -0.5],[-0.5,0, 0.5],[0.5,0, 0.5]],
    [[-0.5,0, -0.5],[0.5,0, -0.5],[0.5,0, 0.5],[-0.5,0, 0.5]],
    [[-0.5,-0.5,0],[0.5,-0.5,0],[0.5,0.5,0],[-0.5,0.5,0]],
    [[ 0.5,-0.5,0],[-0.5,-0.5,0],[-0.5,0.5,0],[0.5,0.5,0]]])

neighbours = np.array([[1.,0.,0.],[-1.,0.,0.],[0.,1.,0.],[0.,-1.,0.],[0.,0.,1.],[0.,0.,-1.]])

In [ ]:
class SphereVolume:
    def __init__(self,_center,_radius):
        self.center = _center
        self.radius = _radius
    def eval(self,p):
        return norm(p-self.center) - self.radius
    def grad(self,p):
        v = p - self.center
        return v / norm(v)

### The XForm Class

The class below is used to transform between the 3D grid and the domain in which your output should live. For instance, say you have a unit sphere represented as a distance field. Perhaps you want the unit sphere to exist in the domain [-1,1]^3 but use a grid of 16x16x16 polygonization cells. In this case the point [0,0,0] in the voxel grid should map to [-1,-1,-1] in the domain and [16,16,16] should map to [1,1,1]. The `XForm` class takes care of this mapping and the inverse.


In [ ]:
class XForm:
    def __init__(self,Plow,Phigh,dims):
        self.dims = np.array(dims)
        self.Plow = np.array(Plow)
        self.Phigh = np.array(Phigh)
        self.scaling = (self.Phigh - self.Plow) / self.dims
    def map(self,pi):
        return self.scaling * np.array(pi) + self.Plow

### Part 1: Finish the polygonizer
You need to find the cube faces that should be generated and also project vertices back onto the isosurface

In [ ]:
def smooth(m):
    # You now the drill - insert code for a gentle Laplacian smoothing
    pos = m.positions()
    new_pos = np.zeros_like(pos)

    for v in m.vertices():
        neighbors_v = list(m.circulate_vertex(v, mode='v'))

        if neighbors_v:
            avg = np.mean(pos[neighbors_v], axis=0)
            new_pos[v] = pos[v] + 0.5 * (avg - pos[v])
        else:
            new_pos[v] = pos[v]

    pos[:] = new_pos
    

def polygonize_volume(vol,xform,tau):
    m = hmesh.Manifold()
    for pi in np.ndindex(tuple(xform.dims)):
        if vol.eval(xform.map(pi)) < tau:
            # Insert code here: For each of the six cube faces, check if neighbor is on the
            # other side of isovalue and generate face if so.
            for i in range(6):
                pn = pi + neighbours[i].astype(int)

                if np.any(pn < 0) or np.any(pn >= xform.dims):
                    continue

                if vol.eval(xform.map(pn)) >= tau:
                    face = [
                        xform.map(np.array(pi, dtype=float) + corner)
                        for corner in cube_faces[i]
                    ]
                    m.add_face(face)
            
    hmesh.stitch(m)
    m.cleanup()
    pos = m.positions()
    smooth(m)
    for v in m.vertices():
        p = xform.map(pos[v])
        # Insert code here: Project point back onto isosurface
        f = vol.eval(p) - tau
        g = vol.grad(p)
        gg = np.dot(g, g)

        if gg > 1e-12:
            p -= (f / gg) * g

        pos[v] = (p - xform.Plow) / xform.scaling

    return m

In [ ]:
vol = SphereVolume([0.,0.5,0.],1.4)
xform = XForm([-1.5,-1.5,-1.5],[1.5,1.5,1.5],[16,16,16])
m = polygonize_volume(vol, xform, 0)
jd.display(m)

### Part 2: Create Gaussian and Gradient of Gaussian functions

In [ ]:
### Exercise: Create functions that return value of 3D Gaussian and gradient. 
ssq = 0.1**2
def Gaussian(p0,p):
    # Compute and return gaussian
    d = np.array(p) - np.array(p0)
    return exp(-np.dot(d, d) / (2 * ssq))

def GaussianGrad(p0,p):
    # Compute and return gradient of gaussian
    d = np.array(p) - np.array(p0)
    return -(d / ssq) * Gaussian(p0, p)


In [ ]:
class Blobs:
    def __init__(self,centers):
        self.centers = centers
    def eval(self,pp):
        p = np.divmod(pp,1)[1]
        return sum([Gaussian(c, p) for c in self.centers])
    def grad(self,pp):
        p = np.divmod(pp,1)[1]
        return sum([GaussianGrad(c, p) for c in self.centers])

### Part 3: play with Blobs
Finally, when you see the blobby cube, try tweaking it to model something different.

In [ ]:
cntrs = [array([0.4,0.4,0.4]) + array([0.2*p[0],0.2*p[1],0.2*p[2]]) for p in np.ndindex((2,2,2))]
vol = Blobs(cntrs)
xform = XForm([0,0,0],[1,1,1],[32,32,32])
m = polygonize_volume(vol, xform, 0.65)
hmesh.flip_orientation(m)
jd.display(m)

## Questions

- What is a distance field and how does it relate to a scalar field? \
A distance field is a function that assigns a distance value to every point in the space which then typically measures the distance to the neareast surface. If it is a signed distance field then the positive and negative values indicate which side of a surface the points lies on. A negative value is inside the object, zero is on the surface and positive values are outside the object. \
Scalar field is a more general function that assigns a single numerical value to every point in the space, making a distance field a type of a scalar field.
- Explain the difference between primal and dual isocontouring methods. \
Primal isocontoring methods extract a surface by examining the original grid's cells and determinig where the isosurface intersects their edges and faces (example Marching Cubes). \
Dual isocontouring methods place mesh vertices in grid cells that intersect the isosurface and connect those vertices across shared cell boundaries (example Dual Contouring). \
The main difference is where the output mesh vertices are place. For primal the vertices are typically placed along the grid edges where the scalar field crosses the isovalue. For Dual the vertices are placed within the grid cells and neigboring cells are connected to form the surfaces. 
- Explain the method for projecting points onto the isocontour. \
This is done by the points being projected back onto the isosurface using the scalar field's gradients. The point is moved along the gradient until the field value matches the desired isovalue. 